# Random Grid Search

Baseline optimizer. See `docs/research.md` for methodology.

## Setup

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

In [ ]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2v2.pmxt.dev",
        "archive:r2.pmxt.dev",
    ),
)

# Example joint-portfolio basket. Swap these slugs for whatever market basket
# you want to evaluate before running the notebook.
BASE_REPLAYS = (
    QuoteReplay(market_slug="human-moon-landing-in-2026", token_index=0),
    QuoteReplay(market_slug="new-coronavirus-pandemic-in-2026", token_index=0),
    QuoteReplay(
        market_slug="will-openais-market-cap-be-between-750b-and-1t-at-market-close-on-ipo-day",
        token_index=0,
    ),
    QuoteReplay(market_slug="okx-ipo-in-2026", token_index=0),
    QuoteReplay(market_slug="nothing-ever-happens-2026", token_index=0),
)

# Joint-portfolio walk-forward split over the ~6-week PMXT window: three
# training slices followed by a short out-of-sample holdout.
TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="train-mar-first-half",
        start_time="2026-03-01T00:00:00Z",
        end_time="2026-03-14T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="train-mar-second-half",
        start_time="2026-03-15T00:00:00Z",
        end_time="2026-03-28T23:59:59Z",
    ),
    ParameterSearchWindow(
        name="train-late-mar-early-apr",
        start_time="2026-03-29T00:00:00Z",
        end_time="2026-04-07T23:59:59Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="holdout-apr-close",
        start_time="2026-04-08T00:00:00Z",
        end_time="2026-04-11T23:59:59Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

PARAMETER_GRID = {
    "fast_period": (32, 64, 96),
    "slow_period": (128, 256, 384),
    "entry_buffer": (0.00025, 0.0005),
    "take_profit": (0.005, 0.01),
    "stop_loss": (0.005, 0.01),
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

MAX_TRIALS = 18
HOLDOUT_TOP_K = 5

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replays=BASE_REPLAYS,
    strategy_spec=STRATEGY_SPEC,
    parameter_grid=PARAMETER_GRID,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    sampler="random",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA random-grid search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured random-grid joint-portfolio search: {MAX_TRIALS} trials over "
    f"{len(BASE_REPLAYS)} market(s), {len(TRAIN_WINDOWS)} train window(s), "
    f"{len(HOLDOUT_WINDOWS)} holdout window(s)."
)

## Run optimizer

In [ ]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

## Leaderboard

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {EXPERIMENT.name} (random-grid)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

## Combined holdout chart

In [ ]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting._prediction_market_backtest import MarketReportConfig
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)
from prediction_market_extensions.backtesting.prediction_market import finalize_market_results

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

COMBINED_PLOT_PANELS = (
    "total_equity",
    "equity",
    "market_pnl",
    "periodic_pnl",
    "yes_price",
    "allocation",
    "total_drawdown",
    "drawdown",
    "total_rolling_sharpe",
    "rolling_sharpe",
    "total_cash_equity",
    "cash_equity",
    "monthly_returns",
    "total_brier_advantage",
    "brier_advantage",
)

COMBINED_REPORT = MarketReportConfig(
    count_key="quotes",
    count_label="Quotes",
    pnl_label="PnL (USDC)",
    market_key="slug",
    summary_report=True,
    summary_report_path=f"output/{NAME}_holdout_joint_portfolio.html",
    summary_plot_panels=COMBINED_PLOT_PANELS,
)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    return_summary_series=True,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()
    finalize_market_results(
        name=backtest.name,
        results=results,
        report=COMBINED_REPORT,
        multi_replay_mode="joint_portfolio",
    )

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
num_markets = len(results)
print(f"Replayed {num_markets} joint-portfolio market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

# Holdout replay stays in joint-portfolio mode. The summary chart overlays
# market-level series in the non-total panels and keeps portfolio totals in the
# total panels.
del results, backtest, updated_html, html_snapshot
gc.collect()